# Dự án 2: AI cho một vấn đề ở địa phương em
**Mở hộp đen AI · Tập 2 · Notebook mẫu**

Tên nhóm: ...  
Thành viên và phân công: ...  
Vấn đề ở địa phương: ...  
Người sẽ dùng sản phẩm: ...

Notebook chạy sẵn với **dữ liệu mẫu do nhóm tác giả tự soạn**: 48 câu phản ánh tưởng tượng gửi tới một phường, chia 4 loại. Nhóm thay bằng dữ liệu của mình ở ô 1. Mọi ô có chữ `TODO` là chỗ nhóm cần sửa.
Ebook Bên trong cỗ máy Trí tuệ nhân tạo - Tập 2. Phát hành tại https://montinhoc.vn

## Phần 1. Dữ liệu

In [ ]:
import pandas as pd

mau = {
 "rác thải": ["rác để ở đầu ngõ ba ngày chưa ai thu", "xe rác đến muộn nên rác tràn ra đường",
   "có người đổ trộm rác xây dựng cạnh hồ", "thùng rác chợ bị vỡ nắp bốc mùi",
   "túi ni lông bay đầy bãi đất trống", "rác thải sinh hoạt vứt xuống kênh",
   "điểm tập kết rác gần trường học rất hôi", "cần thêm thùng phân loại rác ở công viên",
   "rác sau phiên chợ đêm không được dọn", "mùi hôi từ bãi rác lan sang khu dân cư",
   "người dân đốt rác gây khói", "xin lịch thu gom rác cồng kềnh"],
 "giao thông": ["đèn tín hiệu ở ngã tư bị hỏng", "ổ gà lớn trên đường vào trường",
   "xe máy đỗ kín vỉa hè không có lối đi", "biển báo tốc độ bị cây che khuất",
   "giờ tan học ùn tắc trước cổng trường", "vạch kẻ đường cho người đi bộ đã mờ",
   "xe tải chạy nhanh qua khu dân cư", "đường ngập sau mưa xe chết máy",
   "cần thêm gờ giảm tốc ở đoạn cua", "xe buýt không dừng đúng điểm đón",
   "ô tô đỗ chắn lối ra vào ngõ", "đường thiếu đèn chiếu sáng buổi tối"],
 "điện nước": ["mất nước từ sáng chưa có lại", "nước máy có màu vàng và mùi lạ",
   "đường ống nước bị vỡ chảy lênh láng", "mất điện nhiều lần trong tuần",
   "dây điện võng thấp nguy hiểm", "cột điện nghiêng sau trận bão",
   "áp lực nước yếu tầng trên không có nước", "đồng hồ nước chạy dù không dùng",
   "cầu dao tổng của dãy trọ hay nhảy", "nước giếng khoan có mùi phèn",
   "điện chập chờn làm hỏng tủ lạnh", "ống thoát nước bị tắc tràn ra ngõ"],
 "cây xanh": ["cây cổ thụ có cành khô sắp gãy", "cây đổ chắn ngang đường sau mưa",
   "cần cắt tỉa cây che đèn đường", "rễ cây làm nứt vỉa hè",
   "xin trồng thêm cây bóng mát ở sân chơi", "cây trong công viên bị chặt trộm",
   "cỏ mọc cao ở bãi đất trống có rắn", "cành cây chạm vào dây điện",
   "bồn hoa trước nhà văn hoá bị phá", "cây non mới trồng bị chết khô",
   "lá cây rụng làm tắc rãnh thoát nước", "cần tưới cây trong mùa khô"],
}
df = pd.DataFrame([(cau, nhan) for nhan, ds in mau.items() for cau in ds],
                  columns=["noi_dung", "loai"])
# TODO: thay bằng dữ liệu của nhóm: df = pd.read_csv("du_lieu_nhom.csv")
print(df.shape)
print(df.loai.value_counts())

## Phần 2. Mô hình và mốc cơ sở
Từng tiếng và cặp tiếng đứng cạnh nhau (`ngram_range=(1, 2)`) để “xe máy”, “cây xanh” được đếm như một đặc trưng (Bài 12).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

X_hoc, X_thi, y_hoc, y_thi = train_test_split(df.noi_dung, df.loai,
    test_size=0.25, random_state=0, stratify=df.loai)
mo_hinh = make_pipeline(CountVectorizer(ngram_range=(1, 2)),
                        LogisticRegression(max_iter=1000))
mo_hinh.fit(X_hoc, y_hoc)
dung = mo_hinh.score(X_thi, y_thi)
moc = y_thi.value_counts().max() / len(y_thi)   # luôn đoán loại nhiều nhất
print(f"Đúng trên tập thi: {dung:.0%} ({len(y_thi)} câu)")
print(f"Mốc cơ sở: {moc:.0%}")

## Phần 3. Sai ở đâu?

In [ ]:
from sklearn.metrics import confusion_matrix

nhan = sorted(df.loai.unique())
bang = confusion_matrix(y_thi, mo_hinh.predict(X_thi), labels=nhan)
print(pd.DataFrame(bang, index=nhan, columns=nhan))

## Phần 4. Thử câu mới và biết lúc nào nên hỏi người
Nếu mô hình không chắc (xác suất cao nhất dưới 0,5), chuyển cho cán bộ đọc.

In [ ]:
cau_moi = ["nắp cống bị mất rất nguy hiểm cho xe máy",
           "cây xanh ven đường bị sâu ăn trụi lá",
           "hàng xóm mở nhạc to lúc nửa đêm"]
for cau, p in zip(cau_moi, mo_hinh.predict_proba(cau_moi)):
    tot_nhat = p.argmax()
    loai = mo_hinh.classes_[tot_nhat] if p[tot_nhat] >= 0.5 else "CHUYỂN NGƯỜI ĐỌC"
    print(f"{p[tot_nhat]:.2f}  {loai:16}  {cau}")

## Phần 5. Trang demo (mở rộng, cần Internet)
Gradio tạo một trang web nhỏ để người dùng thử mô hình. `share=True` cho một đường link dùng được trong 72 giờ. Chỉ chia sẻ link với người dùng thử đã đồng ý, và **không** cho người dùng nhập thông tin cá nhân.

In [ ]:
!pip install -q gradio
import gradio as gr

def phan_loai(cau):
    p = mo_hinh.predict_proba([cau])[0]
    return {loai: float(x) for loai, x in zip(mo_hinh.classes_, p)}

gr.Interface(fn=phan_loai, inputs=gr.Textbox(label="Nội dung phản ánh"),
             outputs=gr.Label(num_top_classes=3),
             title="Phân loại phản ánh (bản thử của nhóm ...)").launch(share=True)

## Phần 6. Thử với người dùng thật
| Người thử | Họ nhập gì | Mô hình trả lời | Đúng không | Họ góp ý gì |
|---|---|---|---|---|
| | | | | |

## Phần 7. Thẻ mô hình
| Mục | Nội dung |
|---|---|
| Mô hình làm gì | |
| Dữ liệu học (ai, bao nhiêu, thu thế nào) | |
| Độ tốt (thước đo nào, trên tập nào, so mốc cơ sở) | |
| Tốt với ai, kém với ai | |
| Rủi ro: dữ liệu cá nhân, nội dung giả, thiên lệch | |
| Không nên dùng vào việc gì | |
| Khi nào chuyển cho người | |
| Người chịu trách nhiệm | |

## Nhật ký dùng AI của nhóm
| Buổi | Dùng AI vào việc gì | AI sai ở đâu | Nhóm đã sửa thế nào |
|---|---|---|---|
| | | | |